# Pipeline v2: hybrid TF-IDF + Siamese, sweep topic top-k

So với `test-pipeline.ipynb`, notebook này thêm:

1. **Sweep `TOPIC_TOPK_GRID`** (mặc định **3 và 4**) cho bước lọc `macro_domain`.
2. **Hybrid xếp hạng** trên pool đã lọc: `score = alpha * norm(TF-IDF) + (1-alpha) * norm(Siamese cosine)`, chuẩn hoá **min–max theo từng query** trên tập ứng viên.
3. **Logging**: kích thước pool (mean/median), tỷ lệ gold ngoài pool theo từng `topic_topk`, bảng metric cho Siamese thuần và từng `alpha`.

Baselines **TF-IDF full corpus** và **TF-IDF + lọc topic** (lặp theo grid) giữ nguyên ý nghĩa so sánh.

**Phụ thuộc:** `torch`, `datasets`, `numpy`, `tqdm`, `scikit-learn`; tokenizer `model/tokenizer.py`.


In [1]:
from __future__ import annotations

import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, concatenate_datasets
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

PYVI_IMPORT_ERROR = None
UNDERTHESEA_IMPORT_ERROR = None

try:
    from pyvi import ViTokenizer
except ImportError as exc_pyvi:
    PYVI_IMPORT_ERROR = exc_pyvi
    ViTokenizer = None

try:
    from underthesea import word_tokenize
except ImportError as exc_uts:
    UNDERTHESEA_IMPORT_ERROR = exc_uts
    word_tokenize = None


def detect_data_dir() -> Path:
    """Match train-textcnn behavior: detect data_ready by qa_train presence."""
    cwd = Path.cwd().resolve()

    candidates = [
        Path('/kaggle/input/datasets/hngphtrn/legals/data/data_ready'),
        Path('/kaggle/input/datasets/hngphtrn/legals/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data_ready'),
        Path('/kaggle/working/vnlegal-rag/data/data_ready'),
        Path('/kaggle/working/legals/data/data_ready'),
        Path('/kaggle/working/data/data_ready'),
        cwd / 'data' / 'data_ready',
        cwd.parent / 'data' / 'data_ready',
        cwd.parent.parent / 'data' / 'data_ready',
        Path('data/data_ready'),
        Path('../data/data_ready'),
    ]

    try:
        file_parent = Path(__file__).resolve().parent
        candidates.extend([
            file_parent / 'data' / 'data_ready',
            file_parent.parent / 'data' / 'data_ready',
            file_parent.parent.parent / 'data' / 'data_ready',
        ])
    except Exception:
        pass

    seen = set()
    for p in candidates:
        p = p.resolve()
        if str(p) in seen:
            continue
        seen.add(str(p))
        if p.exists() and (p / 'qa_train.csv').is_file():
            return p

    raise FileNotFoundError(
        'Cannot locate data_ready containing qa_train.csv. '
        f'Tried: {[str(x) for x in candidates]}'
    )


DATA_READY_DIR = detect_data_dir()
# Prefer full corpus; fallback to split corpus files if needed.
CORPUS_READY = DATA_READY_DIR / "corpus_full.csv"
CORPUS_TRAIN_READY = DATA_READY_DIR / "corpus_train.csv"
CORPUS_VAL_READY = DATA_READY_DIR / "corpus_val.csv"
CORPUS_TEST_READY = DATA_READY_DIR / "corpus_test.csv"

REPO_ROOT = DATA_READY_DIR.parent.parent if DATA_READY_DIR.parent.name == 'data' else Path.cwd().resolve()
QA_TRAIN_READY = DATA_READY_DIR / "qa_train.csv"
QA_VAL_READY = DATA_READY_DIR / "qa_val.csv"
QA_TEST_READY = DATA_READY_DIR / "qa_test.csv"
LABEL_MAPS = DATA_READY_DIR / "label_maps.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if IS_CUDA:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

# Split train/val/test đã được cố định sẵn trong data_ready (qa_train/qa_val/qa_test),
# nên không dùng các ratio split ở notebook này.


In [2]:
# Split ratio không dùng vì dữ liệu đã split sẵn trong data_ready.
# Giá trị mặc định cho chữ ký hàm; sweep dùng TOPIC_TOPK_GRID
TOPIC_TOPK = 3
RETRIEVE_K = 10

# --- V2: sweep topic + hybrid TF-IDF / Siamese ---
TOPIC_TOPK_GRID = [3, 4]
HYBRID_ALPHA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
VERBOSE_RUN_LOG = True

# Load artifacts: uu tien input/working tren Kaggle, fallback local model/
ARTIFACT_ROOT = DATA_READY_DIR.parent.parent / "artifacts" if DATA_READY_DIR.parent.name == "data" else REPO_ROOT / "artifacts"
MODEL_DIR = REPO_ROOT / "model"
TEXTCNN_ARTIFACT_CANDIDATES = [
    ARTIFACT_ROOT / "textcnn_artifacts",
    Path('/kaggle/input/datasets/hngphtrn/textcnn-artifacts'),
    Path('/kaggle/input/datasets/hngphtrn/textcnn-artifacts'),
    MODEL_DIR / "textcnn_artifacts",
]
SIAMESE_ARTIFACT_CANDIDATES = [
    ARTIFACT_ROOT / "siamese_lstm_traditional_cosine_artifacts",
    Path('/kaggle/input/datasets/hngphtrn/siamese-artifacts'),
    Path('/kaggle/input/datasets/hngphtrn/legals/siamese_artifacts'),
    MODEL_DIR / "siamese_lstm_traditional_cosine_artifacts",
]
TEXTCNN_ARTIFACT_DIR = next((p for p in TEXTCNN_ARTIFACT_CANDIDATES if p.exists()), TEXTCNN_ARTIFACT_CANDIDATES[0])
SIAMESE_ARTIFACT_DIR = next((p for p in SIAMESE_ARTIFACT_CANDIDATES if p.exists()), SIAMESE_ARTIFACT_CANDIDATES[0])

EMBED_DIM = 200
CNN_MAX_LEN = 64  # se duoc ghi de tu textcnn_meta.json
DOC_MAX_LEN = 256
TRIPLET_MARGIN = 0.3
CONTRASTIVE_TEMP = 0.07
SIAMESE_TRIPLET_WEIGHT = 0.5
HARD_NEG_TOPK = 50
STRATIFY_MIN_GROUP = 20  # tối thiểu mẫu mỗi nhóm khi báo metric theo difficulty / question_type

CNN_BATCH_SIZE = 128 if IS_CUDA else 64
TRIPLET_BATCH_SIZE = 64 if IS_CUDA else 32
SIAMESE_EPOCHS = 3 if IS_CUDA else 2
ENCODE_BATCH_SIZE = 128 if IS_CUDA else 64
# Notebook/Kaggle thường lỗi multiprocessing DataLoader khi worker > 0.
NUM_WORKERS = 0

autocast_device = "cuda" if IS_CUDA else "cpu"

print("REPO_ROOT:", REPO_ROOT)
print("DEVICE:", DEVICE)
print("DATA_READY_DIR:", DATA_READY_DIR)
print(
    "corpus_full exists:", CORPUS_READY.is_file(),
    "| corpus splits:", CORPUS_TRAIN_READY.is_file(), CORPUS_VAL_READY.is_file(), CORPUS_TEST_READY.is_file(),
    "| QA train/val/test:", QA_TRAIN_READY.is_file(), QA_VAL_READY.is_file(), QA_TEST_READY.is_file(),
)
if not (QA_TRAIN_READY.is_file() and QA_VAL_READY.is_file() and QA_TEST_READY.is_file()):
    raise FileNotFoundError("Thiếu qa_train/qa_val/qa_test trong data_ready")
if not (CORPUS_READY.is_file() or (CORPUS_TRAIN_READY.is_file() and CORPUS_VAL_READY.is_file() and CORPUS_TEST_READY.is_file())):
    raise FileNotFoundError("Thiếu corpus_full.csv hoặc bộ corpus_train/val/test.csv trong data_ready")

REPO_ROOT: C:\Users\hung\Documents\GitHub\vnlegal-rag
DEVICE: cpu
DATA_READY_DIR: C:\Users\hung\Documents\GitHub\vnlegal-rag\data\data_ready
corpus_full exists: True | corpus splits: True True True | QA train/val/test: True True True


In [3]:
# TextCNN + Siamese được huấn luyện với simple_tokenize (regex \\w+, model/tokenizer.py).
# Không dùng pyvi/underthesea ở inference — tránh lệch token so với checkpoint.
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from model.tokenizer import simple_tokenize


def tokenize_vi_fixed(text: str) -> list[str]:
    return simple_tokenize(text)


corpus_df = pd.read_csv(CORPUS_READY, sep="\t")
qa_train_df = pd.read_csv(QA_TRAIN_READY, sep="\t")
qa_val_df = pd.read_csv(QA_VAL_READY, sep="\t")
qa_test_df = pd.read_csv(QA_TEST_READY, sep="\t")

corpus = Dataset.from_pandas(corpus_df, preserve_index=False)
qa_train = Dataset.from_pandas(qa_train_df, preserve_index=False)
qa_val = Dataset.from_pandas(qa_val_df, preserve_index=False)
qa_test = Dataset.from_pandas(qa_test_df, preserve_index=False)
qa = concatenate_datasets([qa_train, qa_val, qa_test])

passage_list = corpus["passage_id"]
passage_to_idx = {pid: i for i, pid in enumerate(passage_list)}
num_passages = len(passage_list)

articles = corpus["article_content"]
macro_domains = corpus["macro_domain"]

domain_names = sorted(set(macro_domains))
domain_to_id = {d: i for i, d in enumerate(domain_names)}
num_topics = len(domain_names)

print(
    "Số đoạn corpus:",
    num_passages,
    "| QA train:",
    len(qa_train),
    "QA val:",
    len(qa_val),
    "QA test:",
    len(qa_test),
    "| Số chủ đề (macro_domain):",
    num_topics,
)


Số đoạn corpus: 9715 | QA train: 23311 QA val: 2841 QA test: 2991 | Số chủ đề (macro_domain): 8


In [4]:
# Dùng split có sẵn từ data_ready
train_indices = list(range(len(qa_train)))
val_indices = list(range(len(qa_train), len(qa_train) + len(qa_val)))
test_indices = list(range(len(qa_train) + len(qa_val), len(qa)))

train_idx_set = set(train_indices)
val_rows = qa_val
test_rows = qa_test
print("Train QA:", len(train_indices), "Val QA:", len(val_rows), "Test QA:", len(test_rows))

Train QA: 23311 Val QA: 2841 Test QA: 2991


## Đánh giá retrieval

Cell tiếp theo giữ các hàm đánh giá (Recall@K, Precision@K, nDCG@K, MRR, MAP, MR_hit/MedR_hit) cho pipeline TextCNN lọc topic + Siamese / hybrid (v2 dùng `TOPIC_TOPK_GRID`).

In [5]:
def eval_retrieval(rank_fn, eval_rows_iter, ks=(1, 5, 10), desc="Eval"):
    """Đánh giá retrieval với công thức chuẩn cho Recall/Precision/nDCG/MRR/MAP."""
    k_max = max(ks)
    recalls = {kk: [] for kk in ks}
    precs = {kk: [] for kk in ks}
    ndcgs = {kk: [] for kk in ks}
    rr, aps = [], []
    ranks_hit = []
    n_skip = 0

    for row in tqdm(eval_rows_iter, desc=desc, disable=not VERBOSE_RUN_LOG):
        gold = row["passage_id"]
        rel_list = row.get("relevant_passage_ids")
        relevant = set(rel_list) if rel_list else {gold}
        relevant.add(gold)

        if not all(pid in passage_to_idx for pid in relevant):
            n_skip += 1
            continue

        idxs, _ = rank_fn(row["question"], k_max)
        pred = [passage_list[i] for i in idxs[:k_max]]
        rel_flags = [1 if pid in relevant else 0 for pid in pred]

        first_rel_rank = None
        for i, is_rel in enumerate(rel_flags):
            if is_rel:
                first_rel_rank = i + 1
                break
        if first_rel_rank is not None:
            ranks_hit.append(first_rel_rank)

        for kk in ks:
            top_rel = rel_flags[:kk]
            hit = any(top_rel)
            recalls[kk].append(1.0 if hit else 0.0)
            precs[kk].append(float(sum(top_rel)) / float(kk))

            if top_rel:
                dcg = sum(rel / np.log2(i + 2.0) for i, rel in enumerate(top_rel))
                ideal_rel = min(len(relevant), kk)
                idcg = sum(1.0 / np.log2(i + 2.0) for i in range(ideal_rel))
                ndcgs[kk].append(float(dcg / idcg) if idcg > 0.0 else 0.0)
            else:
                ndcgs[kk].append(0.0)

        if first_rel_rank is not None:
            rr.append(1.0 / first_rel_rank)
        else:
            rr.append(0.0)

        hit_count = 0
        prec_sum = 0.0
        for i, is_rel in enumerate(rel_flags, start=1):
            if is_rel:
                hit_count += 1
                prec_sum += hit_count / i
        ap = (prec_sum / len(relevant)) if len(relevant) > 0 else 0.0
        aps.append(ap)

    n = len(recalls[ks[0]])
    out = {f"Recall@{kk}": float(np.mean(recalls[kk])) for kk in ks}
    out.update({f"Precision@{kk}": float(np.mean(precs[kk])) for kk in ks})
    out.update({f"nDCG@{kk}": float(np.mean(ndcgs[kk])) for kk in ks})
    out["MRR"] = float(np.mean(rr)) if n else 0.0
    out["MAP"] = float(np.mean(aps)) if n else 0.0
    out["n_queries"] = float(n)
    out["n_skipped_no_passage"] = float(n_skip)
    if ranks_hit:
        out["MR_hit"] = float(np.mean(ranks_hit))
        out["MedR_hit"] = float(np.median(ranks_hit))
    else:
        out["MR_hit"] = float("nan")
        out["MedR_hit"] = float("nan")
    return out


def eval_retrieval_stratified(rank_fn, eval_dataset: Dataset, ks=(1, 5, 10), min_group: int = 20):
    """Theo difficulty / question_type (mota.md: phân tích theo nhóm)."""
    rows = [r for r in eval_dataset if r["passage_id"] in passage_to_idx]
    out = {}
    for field in ("difficulty", "question_type"):
        groups = {}
        for r in rows:
            groups.setdefault(r.get(field) or "unknown", []).append(r)
        out[field] = {}
        for name in sorted(groups):
            g = groups[name]
            if len(g) < min_group:
                continue
            out[field][name] = eval_retrieval(
                rank_fn, g, ks=ks, desc=f"Eval {field}={name}",
            )
    return out


def print_stratified(title: str, strat: dict):
    print(f"\n=== {title} (stratified) ===")
    for field, groups in strat.items():
        print(f"\n-- {field} --")
        for gname, m in groups.items():
            print(f"  [{gname}] n={int(m.get('n_queries', 0))}", end="")
            for key in sorted(k for k in m if k.startswith(("Recall", "Precision", "nDCG", "MRR", "MAP"))):
                print(f"  {key}={m[key]:.4f}", end="")
            print()



## Cải tiến: TextCNN lọc topic → Siamese / hybrid TF-IDF

V2: sweep **`TOPIC_TOPK_GRID`** (mặc định 3 và 4), thêm **hybrid** min–max trên pool. Checkpoint Siamese có thể khác `mota.md` (xem `siamese_meta.json`).

Luồng: (1) **TextCNN** → top-k chủ đề. (2) Lọc corpus theo `macro_domain`. (3) **Siamese** và/hoặc **TF-IDF** (hybrid) xếp hạng trong pool.

In [6]:
import json

# === Load TextCNN artifacts (vocab + meta) ===
TEXTCNN_META = TEXTCNN_ARTIFACT_DIR / "textcnn_meta.json"
TEXTCNN_VOCAB = TEXTCNN_ARTIFACT_DIR / "tokenizer_vocab.json"
TEXTCNN_WEIGHTS = TEXTCNN_ARTIFACT_DIR / "textcnn_best.pt"

if not (TEXTCNN_META.is_file() and TEXTCNN_VOCAB.is_file() and TEXTCNN_WEIGHTS.is_file()):
    raise FileNotFoundError(
        f"Missing TextCNN artifacts in {TEXTCNN_ARTIFACT_DIR}. Need textcnn_meta.json, tokenizer_vocab.json, textcnn_best.pt. Tried: {[str(p) for p in TEXTCNN_ARTIFACT_CANDIDATES]}"
    )

meta_cnn = json.loads(TEXTCNN_META.read_text(encoding="utf-8"))
vocab_cnn = json.loads(TEXTCNN_VOCAB.read_text(encoding="utf-8"))

PAD = meta_cnn.get("pad_token", "<PAD>")
UNK = meta_cnn.get("unk_token", "<UNK>")
CNN_MAX_LEN = int(meta_cnn.get("max_len", CNN_MAX_LEN))

stoi = vocab_cnn["stoi"]
itos_map = vocab_cnn["itos"]
itos = [itos_map[str(i)] for i in range(len(itos_map))]

labels = list(meta_cnn["labels"])
label2id = dict(meta_cnn["label2id"])

print("Loaded TextCNN artifacts:")
print("  vocab_size=", len(itos))
print("  num_labels=", len(labels))
print("  CNN_MAX_LEN=", CNN_MAX_LEN)
print("  tokenizer_backend=", meta_cnn.get("tokenizer_backend"))

# Re-map domain ids to match TextCNN label order
# (Quan trong de topic_probs[passage_domain_id] khop nhan)
domain_names = labels
_domain_set_corpus = set(macro_domains)
_domain_set_model = set(domain_names)
if _domain_set_corpus != _domain_set_model:
    missing_in_corpus = sorted(_domain_set_model - _domain_set_corpus)
    missing_in_model = sorted(_domain_set_corpus - _domain_set_model)
    raise ValueError(
        "macro_domain mismatch between corpus and TextCNN labels. "
        f"missing_in_corpus={missing_in_corpus} missing_in_model={missing_in_model}"
    )

domain_to_id = {d: i for i, d in enumerate(domain_names)}
num_topics = len(domain_names)


def encode_tokens(stoi: dict[str, int], toks: list[str], max_len: int) -> tuple[list[int], list[float]]:
    unk = stoi[UNK]
    ids = [stoi.get(t, unk) for t in toks[:max_len]]
    pad_id = stoi[PAD]
    length = len(ids)
    if length < max_len:
        ids = ids + [pad_id] * (max_len - length)
    mask = [1.0] * length + [0.0] * (max_len - length)
    return ids, mask

Loaded TextCNN artifacts:
  vocab_size= 6227
  num_labels= 8
  CNN_MAX_LEN= 128
  tokenizer_backend= simple_tokenize


In [7]:
class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        num_classes: int,
        filter_sizes=(2, 3, 4),
        num_filters: int = 128,
        dropout: float = 0.6,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in filter_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            h = F.relu(conv(emb))
            pooled.append(h.max(dim=2).values)
        z = torch.cat(pooled, dim=1)
        z = self.dropout(z)
        return self.fc(z)


class SiameseLSTM(nn.Module):
    def __init__(self, embedding: nn.Embedding, hidden_size: int = 256, num_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.embedding = embedding
        self.lstm = nn.LSTM(
            embedding.embedding_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)

    def encode(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)
        emb = self.drop(emb)
        out, _ = self.lstm(emb)  # (B, L, H)
        m = mask.unsqueeze(-1)
        summed = (out * m).sum(dim=1)
        denom = m.sum(dim=1).clamp(min=1e-6)
        v = summed / denom
        return F.normalize(v, p=2, dim=1)

    def forward(self, q, q_mask, d, d_mask):
        return self.encode(q, q_mask), self.encode(d, d_mask)

In [8]:
def batch_encode_questions(
    questions: list[str],
    max_len: int,
    stoi_override: dict[str, int] | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    _stoi = stoi_override or stoi
    ids, masks = [], []
    for q in questions:
        t = tokenize_vi_fixed(q)
        i, m = encode_tokens(_stoi, t, max_len)
        ids.append(i)
        masks.append(m)
    return torch.tensor(ids, dtype=torch.long), torch.tensor(masks, dtype=torch.float32)


def batch_encode_docs(
    doc_texts: list[str],
    max_len: int,
    stoi_override: dict[str, int] | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    _stoi = stoi_override or stoi
    ids, masks = [], []
    for doc in doc_texts:
        t = tokenize_vi_fixed(doc)
        i, m = encode_tokens(_stoi, t, max_len)
        ids.append(i)
        masks.append(m)
    return torch.tensor(ids, dtype=torch.long), torch.tensor(masks, dtype=torch.float32)


# === Load trained TextCNN weights ===

def build_textcnn_from_state_dict(state_dict: dict, num_classes: int) -> TextCNN:
    emb_w = state_dict["embedding.weight"]
    vocab_size = emb_w.shape[0]
    embed_dim = emb_w.shape[1]

    # infer num_filters and filter_sizes from conv weights
    conv_keys = sorted(k for k in state_dict.keys() if k.startswith("convs.") and k.endswith(".weight"))
    if not conv_keys:
        raise ValueError("Invalid TextCNN state_dict: missing conv weights")

    filter_sizes = []
    num_filters = None
    for k in conv_keys:
        w = state_dict[k]
        # w shape: (out_channels, in_channels, kernel_size)
        if num_filters is None:
            num_filters = int(w.shape[0])
        filter_sizes.append(int(w.shape[2]))

    m = TextCNN(
        vocab_size=vocab_size,
        embed_dim=embed_dim,
        num_classes=num_classes,
        filter_sizes=tuple(filter_sizes),
        num_filters=int(num_filters),
        dropout=float(meta_cnn.get("train_strategy", {}).get("dropout", 0.6)),
    )
    return m

state_cnn = torch.load(TEXTCNN_WEIGHTS, map_location=DEVICE)
topic_model = build_textcnn_from_state_dict(state_cnn, num_classes=num_topics).to(DEVICE)
topic_model.load_state_dict(state_cnn)
topic_model.eval()
print("Loaded TextCNN weights from:", TEXTCNN_WEIGHTS)

Loaded TextCNN weights from: C:\Users\hung\Documents\GitHub\vnlegal-rag\model\textcnn_artifacts\textcnn_best.pt


In [9]:
def predict_topic_probs(model: TextCNN, question: str) -> np.ndarray:
    """Phân phối xác suất chủ đề (macro_domain class) cho câu hỏi."""
    xb, _ = batch_encode_questions([question], CNN_MAX_LEN)
    xb = xb.to(DEVICE, non_blocking=IS_CUDA)
    with torch.no_grad():
        logits = model(xb)
        probs = F.softmax(logits, dim=-1)[0].float().cpu().numpy()
    return probs


def predict_topic_topk(model: TextCNN, question: str, k: int = TOPIC_TOPK) -> tuple[list[int], list[float]]:
    probs = predict_topic_probs(model, question)
    k = min(k, probs.shape[0])
    top_idx = np.argsort(-probs)[:k]
    top_vals = probs[top_idx].tolist()
    return top_idx.tolist(), top_vals


passage_domain_ids = torch.tensor([domain_to_id[d] for d in macro_domains], dtype=torch.long, device=DEVICE)
passage_domain_ids_np = passage_domain_ids.cpu().numpy()


def candidate_indices_for_topics(topic_ids: list[int]) -> np.ndarray:
    mask = np.zeros(num_passages, dtype=bool)
    for tid in topic_ids:
        mask |= passage_domain_ids_np == tid
    idxs = np.where(mask)[0]
    if idxs.size == 0:
        return np.arange(num_passages)
    return idxs

In [10]:
# Baseline TF-IDF (cùng tokenizer simple_tokenize)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

_corpus_tfidf_strs = [" ".join(simple_tokenize(str(a))) for a in articles]
_tfidf = TfidfVectorizer(max_features=80_000, min_df=2, ngram_range=(1, 2))
_tfidf_mat = _tfidf.fit_transform(_corpus_tfidf_strs)
_tfidf_mat = normalize(_tfidf_mat, norm="l2", axis=1)
print(f"TF-IDF matrix: {_tfidf_mat.shape[0]} docs × {_tfidf_mat.shape[1]} terms")


def retrieve_tfidf_full(question: str, k: int):
    qv = _tfidf.transform([" ".join(simple_tokenize(str(question)))])
    qv = normalize(qv, norm="l2", axis=1)
    scores = (qv @ _tfidf_mat.T).toarray().ravel()
    kk = min(k, scores.size)
    top = np.argpartition(-scores, kth=kk - 1)[:kk]
    top = top[np.argsort(-scores[top])]
    return top.tolist(), scores[top].tolist()


def retrieve_tfidf_topic_filtered(question: str, k: int, topic_topk: int):
    top_topic_ids, _ = predict_topic_topk(topic_model, question, k=topic_topk)
    cand_idx = candidate_indices_for_topics(top_topic_ids)
    cand_idx = np.asarray(cand_idx, dtype=np.int64)
    if cand_idx.size == 0:
        return [], []
    qv = _tfidf.transform([" ".join(simple_tokenize(str(question)))])
    qv = normalize(qv, norm="l2", axis=1)
    sub = _tfidf_mat[cand_idx]
    scores = (qv @ sub.T).toarray().ravel()
    kk = min(k, scores.size)
    top_local = np.argpartition(-scores, kth=kk - 1)[:kk]
    top_local = top_local[np.argsort(-scores[top_local])]
    return cand_idx[top_local].tolist(), scores[top_local].tolist()


def tfidf_candidate_scores(question: str, cand_idx: np.ndarray) -> np.ndarray:
    """Điểm TF-IDF cosine (L2) trên ứng viên; thứ tự khớp cand_idx."""
    cand_idx = np.asarray(cand_idx, dtype=np.int64)
    if cand_idx.size == 0:
        return np.array([], dtype=np.float64)
    qv = _tfidf.transform([" ".join(simple_tokenize(str(question)))])
    qv = normalize(qv, norm="l2", axis=1)
    sub = _tfidf_mat[cand_idx]
    return (qv @ sub.T).toarray().ravel()


tfidf_full_metrics = eval_retrieval(
    lambda q, kk: retrieve_tfidf_full(q, kk),
    test_rows,
    desc="TF-IDF full corpus",
)
print("Baseline TF-IDF (toàn corpus):", tfidf_full_metrics)

for _tk in TOPIC_TOPK_GRID:
    m = eval_retrieval(
        lambda q, kk, tk=_tk: retrieve_tfidf_topic_filtered(q, kk, topic_topk=tk),
        test_rows,
        desc=f"TF-IDF + TextCNN top-{_tk} topic",
    )
    print(f"TF-IDF + lọc top-{_tk} topic:", m)

for _tk in TOPIC_TOPK_GRID:
    _gold_miss_topic = 0
    _n_topic_chk = 0
    _pool_sizes = []
    for row in test_rows:
        gold = row["passage_id"]
        if gold not in passage_to_idx:
            continue
        gi = passage_to_idx[gold]
        top_topic_ids, _ = predict_topic_topk(topic_model, row["question"], k=_tk)
        cand = candidate_indices_for_topics(top_topic_ids)
        _pool_sizes.append(len(cand))
        _n_topic_chk += 1
        if gi not in set(cand):
            _gold_miss_topic += 1
    pct = 100.0 * _gold_miss_topic / max(_n_topic_chk, 1)
    print(
        f"[topic_topk={_tk}] pool mean={np.mean(_pool_sizes):.0f} "
        f"median={np.median(_pool_sizes):.0f} | gold NGOÀI pool: "
        f"{_gold_miss_topic}/{_n_topic_chk} ({pct:.1f}%)"
    )


TF-IDF matrix: 9715 docs × 80000 terms


TF-IDF full corpus:   0%|          | 0/2991 [00:00<?, ?it/s]

Baseline TF-IDF (toàn corpus): {'Recall@1': 0.5670344366432631, 'Recall@5': 0.8251420929455032, 'Recall@10': 0.8886659979939819, 'Precision@1': 0.5670344366432631, 'Precision@5': 0.1650284185891006, 'Precision@10': 0.08886659979939818, 'nDCG@1': 0.5670344366432631, 'nDCG@5': 0.7076013296597546, 'nDCG@10': 0.728210559498498, 'MRR': 0.6767699925172342, 'MAP': 0.6767699925172342, 'n_queries': 2991.0, 'n_skipped_no_passage': 0.0, 'MR_hit': 2.016553799849511, 'MedR_hit': 1.0}


TF-IDF + TextCNN top-3 topic:   0%|          | 0/2991 [00:00<?, ?it/s]

TF-IDF + lọc top-3 topic: {'Recall@1': 0.5115346038114343, 'Recall@5': 0.7499164159144099, 'Recall@10': 0.8057505850885991, 'Precision@1': 0.5115346038114343, 'Precision@5': 0.14998328318288195, 'Precision@10': 0.0805750585088599, 'nDCG@1': 0.5115346038114343, 'nDCG@5': 0.641555819312873, 'nDCG@10': 0.659653742566305, 'MRR': 0.6127309176205866, 'MAP': 0.6127309176205866, 'n_queries': 2991.0, 'n_skipped_no_passage': 0.0, 'MR_hit': 2.0107883817427386, 'MedR_hit': 1.0}


TF-IDF + TextCNN top-4 topic:   0%|          | 0/2991 [00:00<?, ?it/s]

TF-IDF + lọc top-4 topic: {'Recall@1': 0.5272484119023738, 'Recall@5': 0.7706452691407556, 'Recall@10': 0.8288197927114678, 'Precision@1': 0.5272484119023738, 'Precision@5': 0.1541290538281511, 'Precision@10': 0.08288197927114677, 'nDCG@1': 0.5272484119023738, 'nDCG@5': 0.6599504532761784, 'nDCG@10': 0.6788463110409984, 'MRR': 0.6307037514660383, 'MAP': 0.6307037514660383, 'n_queries': 2991.0, 'n_skipped_no_passage': 0.0, 'MR_hit': 2.010891488503429, 'MedR_hit': 1.0}
[topic_topk=3] pool mean=4113 median=4160 | gold NGOÀI pool: 277/2991 (9.3%)
[topic_topk=4] pool mean=5242 median=5215 | gold NGOÀI pool: 204/2991 (6.8%)


In [11]:
# Auto-search alpha tốt nhất + bảng tổng hợp best-by-metric
# Có fallback để không phụ thuộc chặt vào thứ tự chạy cell.

USE_HYBRID = "retrieve_hybrid_topic" in globals()
if not USE_HYBRID:
    print("[WARN] Thiếu retrieve_hybrid_topic -> fallback sang retrieve_tfidf_topic_filtered (alpha sẽ không có tác dụng).")

def _rank_alpha_search(q, kk, topic_topk, alpha):
    if USE_HYBRID:
        return retrieve_hybrid_topic(q, kk, topic_topk=topic_topk, alpha=alpha)
    return retrieve_tfidf_topic_filtered(q, kk, topic_topk=topic_topk)

ALPHA_SEARCH_GRID = np.round(np.arange(0.0, 1.0001, 0.05), 2).tolist()
METRIC_KEYS = ["Recall@1", "Recall@5", "Recall@10", "nDCG@10", "MRR"]

search_rows = []
for tk in TOPIC_TOPK_GRID:
    for a in ALPHA_SEARCH_GRID:
        m = eval_retrieval(
            lambda q, kk, t=tk, al=float(a): _rank_alpha_search(q, kk, topic_topk=t, alpha=al),
            test_rows,
            desc=f"AlphaSearch top-{tk} a={a:.2f}",
        )
        row = {"topic_topk": int(tk), "alpha": float(a)}
        row.update({k: float(m.get(k, float("nan"))) for k in METRIC_KEYS})
        search_rows.append(row)

alpha_search_df = pd.DataFrame(search_rows).sort_values(["topic_topk", "alpha"]).reset_index(drop=True)

print("=== Alpha search summary (top rows) ===")
print(alpha_search_df.head(12).to_string(index=False))

best_rows = []
for metric in METRIC_KEYS:
    idx = alpha_search_df[metric].idxmax()
    rec = alpha_search_df.loc[idx]
    best_rows.append({
        "metric": metric,
        "best_topic_topk": int(rec["topic_topk"]),
        "best_alpha": float(rec["alpha"]),
        "best_value": float(rec[metric]),
    })

best_by_metric_df = pd.DataFrame(best_rows).sort_values("metric").reset_index(drop=True)
print("\n=== Best-by-metric ===")
print(best_by_metric_df.to_string(index=False))

# Dòng gợi ý cấu hình tốt nhất theo MRR (tie-break: Recall@10)
best_mrr = alpha_search_df.sort_values(["MRR", "Recall@10"], ascending=False).iloc[0]
print(
    "\nKhuyến nghị (ưu tiên MRR): "
    f"topic_topk={int(best_mrr['topic_topk'])}, alpha={best_mrr['alpha']:.2f}, "
    f"MRR={best_mrr['MRR']:.4f}, Recall@10={best_mrr['Recall@10']:.4f}"
)


AlphaSearch top-3 a=0.00:   0%|          | 0/2991 [00:00<?, ?it/s]

NameError: name 'retrieve_hybrid_topic' is not defined

In [ ]:
import json

# === Load trained Siamese artifacts ===
SIAMESE_META = SIAMESE_ARTIFACT_DIR / "siamese_meta.json"
SIAMESE_VOCAB = SIAMESE_ARTIFACT_DIR / "tokenizer_vocab.json"
SIAMESE_WEIGHTS = SIAMESE_ARTIFACT_DIR / "siamese_lstm_traditional_cosine_best.pt"

if not (SIAMESE_META.is_file() and SIAMESE_VOCAB.is_file() and SIAMESE_WEIGHTS.is_file()):
    raise FileNotFoundError(
        f"Missing Siamese artifacts in {SIAMESE_ARTIFACT_DIR}. Need siamese_meta.json, tokenizer_vocab.json, siamese_lstm_traditional_cosine_best.pt"
    )

meta_s = json.loads(SIAMESE_META.read_text(encoding="utf-8"))
vocab_s = json.loads(SIAMESE_VOCAB.read_text(encoding="utf-8"))

SIAMESE_MAX_Q_LEN = int(meta_s.get("max_q_len", CNN_MAX_LEN))
SIAMESE_MAX_D_LEN = int(meta_s.get("max_d_len", DOC_MAX_LEN))

stoi_s = vocab_s["stoi"]
itos_s_map = vocab_s["itos"]
itos_s = [itos_s_map[str(i)] for i in range(len(itos_s_map))]

print("Loaded Siamese artifacts:")
print("  vocab_size=", len(itos_s))
print("  max_q_len=", SIAMESE_MAX_Q_LEN)
print("  max_d_len=", SIAMESE_MAX_D_LEN)

siamese_embedding = nn.Embedding(
    len(itos_s),
    int(meta_s.get("embed_dim", EMBED_DIM)),
    padding_idx=int(stoi_s.get(PAD, 0)),
)
siamese = SiameseLSTM(
    siamese_embedding,
    hidden_size=int(meta_s.get("hidden_size", 256)),
    num_layers=int(meta_s.get("num_layers", 2)),
    dropout=0.3,
).to(DEVICE)

state_s = torch.load(SIAMESE_WEIGHTS, map_location=DEVICE)

if isinstance(state_s, dict) and "state_dict" in state_s:
    state_s = state_s["state_dict"]

# Tuong thich checkpoint cu: key co prefix "encoder." trong khi model hien tai khong co.
if any(k.startswith("encoder.") for k in state_s.keys()):
    state_s = {k.replace("encoder.", "", 1): v for k, v in state_s.items()}

siamese.load_state_dict(state_s)
siamese.eval()
print("Loaded Siamese weights from:", SIAMESE_WEIGHTS)

# Cache doc tensor để encode nhanh hơn.
doc_ids_all, doc_mask_all = batch_encode_docs(list(articles), SIAMESE_MAX_D_LEN, stoi_override=stoi_s)

Loaded Siamese artifacts:
  vocab_size= 6227
  max_q_len= 64
  max_d_len= 256
Loaded Siamese weights from: C:\Users\hung\Documents\GitHub\vnlegal-rag\model\siamese_lstm_traditional_cosine_artifacts\siamese_lstm_traditional_cosine_best.pt


In [ ]:
def _minmax01_per_query(scores: np.ndarray) -> np.ndarray:
    """Min–max trên pool ứng viên (một query)."""
    scores = np.asarray(scores, dtype=np.float64)
    if scores.size == 0:
        return scores
    lo, hi = float(scores.min()), float(scores.max())
    if hi - lo < 1e-12:
        return np.full_like(scores, 0.5)
    return (scores - lo) / (hi - lo)


@torch.no_grad()
def encode_all_passages(model: SiameseLSTM, batch_size: int = ENCODE_BATCH_SIZE) -> np.ndarray:
    vecs = []
    model.eval()
    for start in tqdm(
        range(0, num_passages, batch_size),
        desc="Encode corpus",
        disable=not VERBOSE_RUN_LOG,
    ):
        end = min(start + batch_size, num_passages)
        x = doc_ids_all[start:end].to(DEVICE, non_blocking=IS_CUDA)
        m = doc_mask_all[start:end].to(DEVICE, non_blocking=IS_CUDA)
        with torch.amp.autocast(device_type=autocast_device, enabled=IS_CUDA):
            v = model.encode(x, m)
        vecs.append(v.float().cpu().numpy())
    return np.concatenate(vecs, axis=0)


doc_emb_all = encode_all_passages(siamese)


@torch.no_grad()
def encode_query_vec(question: str) -> np.ndarray:
    x, m = batch_encode_questions([question], SIAMESE_MAX_Q_LEN, stoi_override=stoi_s)
    x = x.to(DEVICE, non_blocking=IS_CUDA)
    m = m.to(DEVICE, non_blocking=IS_CUDA)
    with torch.amp.autocast(device_type=autocast_device, enabled=IS_CUDA):
        v = siamese.encode(x, m)
    return v.float().cpu().numpy()


def retrieve_siamese_topic(question: str, k: int, topic_topk: int):
    top_topic_ids, _ = predict_topic_topk(topic_model, question, k=topic_topk)
    cand_idx = candidate_indices_for_topics(top_topic_ids)
    cand_idx = np.asarray(cand_idx, dtype=np.int64)
    if cand_idx.size == 0:
        return [], []
    qv = encode_query_vec(question)
    sub = doc_emb_all[cand_idx]
    dense_scores = (sub @ qv.T).ravel()
    kk = min(k, dense_scores.size)
    top_local = np.argpartition(-dense_scores, kth=kk - 1)[:kk]
    top_local = top_local[np.argsort(-dense_scores[top_local])]
    return cand_idx[top_local].tolist(), dense_scores[top_local].tolist()


def retrieve_hybrid_topic(question: str, k: int, topic_topk: int, alpha: float):
    """alpha: trọng số TF-IDF (sau min–max). (1-alpha): Siamese (sau min–max)."""
    top_topic_ids, _ = predict_topic_topk(topic_model, question, k=topic_topk)
    cand_idx = candidate_indices_for_topics(top_topic_ids)
    cand_idx = np.asarray(cand_idx, dtype=np.int64)
    if cand_idx.size == 0:
        return [], []
    s_tf = tfidf_candidate_scores(question, cand_idx)
    qv = encode_query_vec(question)
    sub = doc_emb_all[cand_idx]
    s_sia = (sub @ qv.T).ravel()
    s_tf_n = _minmax01_per_query(s_tf)
    s_sia_n = _minmax01_per_query(s_sia)
    hybrid = float(alpha) * s_tf_n + (1.0 - float(alpha)) * s_sia_n
    kk = min(k, hybrid.size)
    top_local = np.argpartition(-hybrid, kth=kk - 1)[:kk]
    top_local = top_local[np.argsort(-hybrid[top_local])]
    return cand_idx[top_local].tolist(), hybrid[top_local].tolist()


def _print_metric_row(label: str, m: dict) -> None:
    keys = ["Recall@1", "Recall@5", "Recall@10", "nDCG@10", "MRR"]
    parts = [f"{label[:68]:<68}"] + [f"{m.get(k, float('nan')):.4f}" for k in keys]
    print(" | ".join(parts))


if VERBOSE_RUN_LOG:
    print("\n=== Cấu hình V2 ===")
    print("TOPIC_TOPK_GRID:", TOPIC_TOPK_GRID)
    print("HYBRID_ALPHA_GRID:", HYBRID_ALPHA_GRID, "(alpha = trọng số TF-IDF sau min–max)")

siamese_metrics_by_topk = {}
if VERBOSE_RUN_LOG:
    print("\n=== Siamese thuần (theo topic_topk) ===")
    print(f"{'config':<68} | Recall@1 | Recall@5 | Recall@10 | nDCG@10 | MRR")
for tk in TOPIC_TOPK_GRID:
    m = eval_retrieval(
        lambda q, kk, t=tk: retrieve_siamese_topic(q, kk, topic_topk=t),
        test_rows,
        desc=f"Siamese top-{tk} topic",
    )
    siamese_metrics_by_topk[tk] = m
    if VERBOSE_RUN_LOG:
        _print_metric_row(f"Siamese | topic_topk={tk}", m)

hybrid_metrics_by_cfg = {}
if VERBOSE_RUN_LOG:
    print("\n=== Hybrid TF-IDF + Siamese (topic_topk × alpha) ===")
    print(f"{'config':<68} | Recall@1 | Recall@5 | Recall@10 | nDCG@10 | MRR")
for tk in TOPIC_TOPK_GRID:
    for a in HYBRID_ALPHA_GRID:
        m = eval_retrieval(
            lambda q, kk, t=tk, al=a: retrieve_hybrid_topic(q, kk, topic_topk=t, alpha=al),
            test_rows,
            desc=f"Hybrid top-{tk} a={a}",
        )
        hybrid_metrics_by_cfg[(tk, a)] = m
        if VERBOSE_RUN_LOG:
            _print_metric_row(f"Hybrid | topic_topk={tk} alpha={a}", m)


Encode corpus:   0%|          | 0/152 [00:00<?, ?it/s]


=== Cấu hình V2 ===
TOPIC_TOPK_GRID: [3, 4]
HYBRID_ALPHA_GRID: [0.0, 0.25, 0.5, 0.75, 1.0] (alpha = trọng số TF-IDF sau min–max)

=== Siamese thuần (theo topic_topk) ===
config                                                               | Recall@1 | Recall@5 | Recall@10 | nDCG@10 | MRR


Siamese top-3 topic:   0%|          | 0/2991 [00:00<?, ?it/s]

Siamese | topic_topk=3                                               | 0.3320 | 0.5603 | 0.6326 | 0.4779 | 0.4289


Siamese top-4 topic:   0%|          | 0/2991 [00:00<?, ?it/s]

...

Hybrid | topic_topk=4 alpha=0.5                                      | 0.5777 | 0.8081 | 0.8623 | 0.7218 | 0.6766


Hybrid top-4 a=0.75:   0%|          | 0/2991 [00:00<?, ?it/s]

Hybrid | topic_topk=4 alpha=0.75                                     | 0.5597 | 0.7991 | 0.8549 | 0.7089 | 0.6620


Hybrid top-4 a=1.0:   0%|          | 0/2991 [00:00<?, ?it/s]

Hybrid | topic_topk=4 alpha=1.0                                      | 0.5272 | 0.7706 | 0.8288 | 0.6788 | 0.6307


### Ghi chú triển khai (v2)

- **Hybrid**: TF-IDF và Siamese cosine được **min–max** trên pool ứng viên rồi trộn `alpha` / `(1-alpha)`. `alpha=1` ≈ TF-IDF trên pool; `alpha=0` ≈ Siamese.
- **TOPIC_TOPK_GRID**: so sánh lọc chủ đề (3 vs 4).
- Baseline **TF-IDF toàn corpus** không qua TextCNN.
- Tokenizer: **`simple_tokenize`** (`model/tokenizer.py`).
